In [ ]:
#%pip install pillow pytesseract

In [1]:
import re
import json
import uuid
import math
import random
from pathlib import Path
from typing import List, Dict, Optional

import fitz  # PyMuPDF
import numpy as np
import pandas as pd
from io import BytesIO
from typing import Optional

import fitz  # PyMuPDF
from PIL import Image
import pytesseract

from sklearn.metrics.pairwise import cosine_similarity

# Embedding model
from sentence_transformers import SentenceTransformer

/home/yash/Dev_Projects/.venv_notebook/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
#from pathlib import Path

BASE_DIR = Path("/home/yash/Dev_Projects")
NOTEBOOK_RECORDS_DIR = BASE_DIR / "notebook_records"
NOTEBOOK_RECORDS_DIR.mkdir(parents=True, exist_ok=True)

print("Notebook records dir:", NOTEBOOK_RECORDS_DIR)

Notebook records dir: /home/yash/Dev_Projects/notebook_records


In [3]:
# =========================
# INPUT PDF
# =========================
PDF_PATH = Path("/home/yash/Dev_Projects/RAG PDF/project/Reports/Annual-Report-2024-25.pdf")

assert PDF_PATH.exists(), f"PDF not found: {PDF_PATH}"

# =========================
# CONFIG
# =========================
TARGET_CHUNK_WORDS = 180
MAX_CHUNK_WORDS = 260
MIN_CHUNK_WORDS = 80
CHUNK_OVERLAP_WORDS = 30

MAX_HEADING_WORDS = 12
MIN_HEADING_UPPER_RATIO = 0.60

TOP_K = 5

EMBEDDING_MODEL_NAME = "all-MiniLM-L6-v2"

random.seed(42)
np.random.seed(42)

In [4]:
# ==========================================
# PHASE 1 — PDF INGESTION + PARSING
# ==========================================

def normalize_spaces(text: str) -> str:
    text = text.replace("\xa0", " ")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def clean_candidate_line(line: str) -> str:
    line = normalize_spaces(line)
    line = re.sub(r"^[^\w]+|[^\w\)\.&,\- ]+$", "", line).strip()
    return line


def extract_report_title(pages: list[dict]) -> Optional[str]:
    first_pages = pages[:5]
    title_candidates = []

    for page in first_pages:
        lines = [clean_candidate_line(x) for x in page["text"].splitlines() if x.strip()]
        for line in lines[:40]:
            line_l = line.lower()
            if "annual report" in line_l or "integrated report" in line_l:
                title_candidates.append(line)

    if title_candidates:
        title_candidates = sorted(title_candidates, key=len)
        return title_candidates[0]

    return None


def extract_financial_year(document_name: str, pages: list[dict]) -> Optional[str]:
    candidates = []

    candidates.extend(re.findall(r"FY[-_ ]?(20\d{2}[-/]\d{2,4})", document_name, flags=re.I))
    candidates.extend(re.findall(r"(20\d{2}[-/]\d{2,4})", document_name, flags=re.I))

    first_pages_text = "\n".join([p["text"] for p in pages[:8]])
    candidates.extend(re.findall(r"FY[- ]?(20\d{2}[-/]\d{2,4})", first_pages_text, flags=re.I))
    candidates.extend(re.findall(r"(20\d{2}[-/]\d{2,4})", first_pages_text, flags=re.I))

    if not candidates:
        return None

    normalized = []
    for c in candidates:
        c = c.upper().replace("FY", "").strip()
        c = re.sub(r"\s+", "", c)
        normalized.append(c)

    normalized.sort(key=lambda x: (0 if re.fullmatch(r"20\d{2}[-/]\d{2}", x) else 1, len(x)))
    return normalized[0]


def render_page_for_ocr(pdf, page_index: int, zoom: float = 3.0) -> Image.Image:
    page = pdf[page_index]
    matrix = fitz.Matrix(zoom, zoom)
    pix = page.get_pixmap(matrix=matrix, alpha=False)
    img = Image.open(BytesIO(pix.tobytes("png")))
    return img


def ocr_page_image(image: Image.Image) -> str:
    text = pytesseract.image_to_string(image, lang="eng")
    return normalize_spaces(text)


def extract_last_page_ocr_text(pdf) -> str:
    """
    OCR the last page first.
    If OCR result is too weak, try the previous 2 pages as fallback.
    """
    total_pages = len(pdf)
    candidate_indices = [total_pages - 1, total_pages - 2, total_pages - 3]

    for idx in candidate_indices:
        if idx < 0:
            continue

        img = render_page_for_ocr(pdf, idx, zoom=3.0)
        text = ocr_page_image(img)

        if len(text.strip()) >= 30:
            return text

    return ""


def is_bad_company_candidate(line: str) -> bool:
    line_l = line.lower().strip()

    bad_exact = {
        "contents",
        "content",
        "index",
        "notice",
        "corporate governance",
        "financial statements",
        "standalone financial statements",
        "consolidated financial statements",
        "management discussion and analysis report",
        "business responsibility and sustainability report",
        "forward-looking statement",
        "working capital loan",
        "financial liabilities",
        "financial assets",
        "bankers",
        "auditors",
        "registrars",
        "investors",
        "annual report",
        "integrated report",
        "director, finance",
        "director finance",
    }

    bad_contains = {
        "annual report",
        "integrated report",
        "contents",
        "content",
        "index",
        "corporate governance",
        "financial statements",
        "standalone financial statements",
        "consolidated financial statements",
        "management discussion",
        "sustainability report",
        "working capital loan",
        "liabilities",
        "liability",
        "assets",
        "asset",
        "equity",
        "borrowings",
        "receivables",
        "payables",
        "cash flow",
        "cash flows",
        "profit and loss",
        "balance sheet",
        "statement of",
        "notes annexed",
        "notes to",
        "director, finance",
        "director finance",
        "chairman",
        "managing director",
        "company secretary",
    }

    if line_l in bad_exact:
        return True

    if any(x in line_l for x in bad_contains):
        return True

    if len(line.split()) < 2:
        return True

    if len(line.split()) > 10:
        return True

    if len(line) > 120:
        return True

    if re.search(r"\b\d{2,}\b", line_l):
        return True

    return False


def score_company_candidate(line: str) -> int:
    score = 0
    line_l = line.lower()

    suffix_words = [
        "limited", "ltd", "bank", "finance", "financial services",
        "capital", "services", "industries", "motors", "technology",
        "technologies", "corporation", "corp", "holdings",
        "company", "enterprises"
    ]

    strong_patterns = [
        r"\blimited\b",
        r"\bltd\b",
        r"\bbank\b",
        r"\bfinance\b",
        r"\bfinancial services\b",
        r"\bcapital\b",
        r"\bservices\b",
        r"\bindustries\b",
        r"\bmotors\b",
        r"\btechnologies\b",
        r"\bcorporation\b",
        r"\bholdings\b",
    ]

    if 2 <= len(line.split()) <= 7:
        score += 2

    if line.istitle():
        score += 2

    if re.fullmatch(r"[A-Za-z0-9&,\.\-() ]+", line):
        score += 1

    if any(word in line_l for word in suffix_words):
        score += 8

    if any(re.search(p, line_l) for p in strong_patterns):
        score += 8

    # Strong preference for realistic company names
    if 2 <= len(line.split()) <= 5:
        score += 2

    return score


def extract_company_name_from_last_page_ocr(ocr_text: str) -> Optional[str]:
    """
    Primary strategy:
    extract company name from OCR text of last page / back pages.
    """
    lines = [clean_candidate_line(x) for x in ocr_text.splitlines() if x.strip()]

    candidates = []
    for line in lines[:80]:
        if is_bad_company_candidate(line):
            continue

        score = score_company_candidate(line)
        if score > 0:
            candidates.append((score, line))

    if not candidates:
        return None

    candidates.sort(key=lambda x: (-x[0], len(x[1])))

    preferred = [
        item for item in candidates
        if any(x in item[1].lower() for x in [
            "limited", "ltd", "bank", "finance", "industries",
            "motors", "corporation", "holdings", "services", "capital"
        ])
    ]

    if preferred:
        return preferred[0][1]

    return candidates[0][1]


# -------- Parse PDF --------
pdf = fitz.open(PDF_PATH)

pages = []
empty_pages = 0

for i in range(len(pdf)):
    page = pdf[i]
    text = page.get_text("text") or ""
    text = text.strip()

    if not text:
        empty_pages += 1

    pages.append({
        "page_number": i + 1,
        "text": text
    })

document_id = uuid.uuid4().hex
document_name = PDF_PATH.name

# -------- OCR from last page / back pages --------
last_page_ocr_text = extract_last_page_ocr_text(pdf)

print("=" * 100)
print("LAST PAGE OCR PREVIEW")
print("=" * 100)
print(last_page_ocr_text[:3000])

company_name = extract_company_name_from_last_page_ocr(last_page_ocr_text)
financial_year = extract_financial_year(document_name, pages)
report_title = extract_report_title(pages)

phase1_output = {
    "document_id": document_id,
    "document_name": document_name,
    "company_name": company_name,
    "financial_year": financial_year,
    "report_title": report_title,
    "total_pages": len(pages),
    "empty_pages": empty_pages,
    "pages": pages,
    "last_page_ocr_text": last_page_ocr_text
}

print("=" * 100)
print("PHASE 1 COMPLETE")
print("=" * 100)
print("Document Name  :", phase1_output["document_name"])
print("Document ID    :", phase1_output["document_id"])
print("Company Name   :", phase1_output["company_name"])
print("Financial Year :", phase1_output["financial_year"])
print("Report Title   :", phase1_output["report_title"])
print("Total Pages    :", phase1_output["total_pages"])
print("Empty Pages    :", phase1_output["empty_pages"])

phase1_save_path = NOTEBOOK_RECORDS_DIR / "phase1_output.json"

with open(phase1_save_path, "w", encoding="utf-8") as f:
    json.dump(phase1_output, f, ensure_ascii=False, indent=2)

print("\nPhase 1 output saved to:", phase1_save_path)

LAST PAGE OCR PREVIEW
(0 Inunrk
HMT Limited

CIN No. L29230KA1953G01000748
HMT Bhavan, No. 59,
Bellary Road, Bengaluru - 560 032
Website : www.hmtindia.com
PHASE 1 COMPLETE
Document Name  : Annual-Report-2024-25.pdf
Document ID    : cac28f9d866d468b9b958ee01d9366e6
Company Name   : HMT Limited
Financial Year : 2024-25
Report Title   : Annual Report 2024 - 25
Total Pages    : 241
Empty Pages    : 3

Phase 1 output saved to: /home/yash/Dev_Projects/notebook_records/phase1_output.json


In [5]:
# ==========================================
# PHASE 2 — CLEANING + SECTIONING + CHUNKING
# ==========================================

def clean_line(line: str) -> str:
    line = line.strip()
    line = re.sub(r"\s+", " ", line)
    return line


def remove_page_artifacts(text: str) -> str:
    cleaned_lines = []

    for raw_line in text.splitlines():
        line = clean_line(raw_line)

        if not line:
            cleaned_lines.append("")
            continue

        if re.fullmatch(r"\d{1,4}", line):
            continue

        if re.fullmatch(r"page\s+\d{1,4}", line, flags=re.I):
            continue

        if re.fullmatch(r"[\*\-–—•\s]{3,}", line):
            continue

        if line.lower().strip() in {
            "annual report",
            "annual report 2024-25",
            "integrated report"
        }:
            continue

        cleaned_lines.append(line)

    return "\n".join(cleaned_lines).strip()


def join_broken_lines(text: str) -> str:
    lines = text.splitlines()
    merged = []
    buffer = []

    def flush_buffer():
        nonlocal buffer, merged
        if buffer:
            merged.append(" ".join(buffer).strip())
            buffer = []

    for line in lines:
        stripped = line.strip()

        if not stripped:
            flush_buffer()
            merged.append("")
            continue

        if len(stripped.split()) <= MAX_HEADING_WORDS and stripped.isupper():
            flush_buffer()
            merged.append(stripped)
            continue

        if re.match(r"^(\-|\•|\*|\d+\.)\s+", stripped):
            flush_buffer()
            merged.append(stripped)
            continue

        buffer.append(stripped)

    flush_buffer()
    text = "\n".join(merged)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def clean_page_text(text: str) -> str:
    text = normalize_spaces(text)
    text = remove_page_artifacts(text)
    text = join_broken_lines(text)
    text = normalize_spaces(text)
    return text


def clean_page_text_for_contents(text: str) -> str:
    text = normalize_spaces(text)
    text = remove_page_artifacts(text)
    return text


# -------- Clean all pages --------
cleaned_pages = []
for p in phase1_output["pages"]:
    cleaned_pages.append({
        "page_number": p["page_number"],
        "text": clean_page_text(p["text"])
    })

# Separate version for contents/index parsing: no line-joining
contents_pages = []
for p in phase1_output["pages"]:
    contents_pages.append({
        "page_number": p["page_number"],
        "text": clean_page_text_for_contents(p["text"])
    })


# -------- Helper Functions for Contents / Index page detection --------
def normalize_for_match(text: str) -> str:
    return re.sub(r"\s+", " ", text.lower()).strip()


# def score_contents_page(text: str) -> int:
#     text_l = normalize_for_match(text)
#     score = 0

#     if re.search(r"\bcontents\b", text_l):
#         score += 8
#     if re.search(r"\bcontent\b", text_l):
#         score += 5
#     if re.search(r"\bindex\b", text_l):
#         score += 6

#     lines = [line.strip() for line in text.splitlines() if line.strip()]

#     dotted_lines = 0
#     numbered_lines = 0

#     for line in lines:
#         if re.search(r"\.{3,}", line):
#             dotted_lines += 1
#         if re.search(r"\b\d{1,3}\s*$", line):
#             numbered_lines += 1

#     score += min(dotted_lines, 15)
#     score += min(numbered_lines, 15)

#     return score

def score_contents_page(text: str) -> int:
    text_l = normalize_for_match(text)
    score = 0

    has_keyword = False

    if re.search(r"\bcontents\b", text_l):
        score += 10
        has_keyword = True
    if re.search(r"\bcontent\b", text_l):
        score += 6
        has_keyword = True
    if re.search(r"\bindex\b", text_l):
        score += 8
        has_keyword = True

    if not has_keyword:
        return 0

    lines = [line.strip() for line in text.splitlines() if line.strip()]
    dotted_lines = sum(1 for line in lines if re.search(r"\.{3,}", line))
    numbered_lines = sum(1 for line in lines if re.search(r"\b\d{1,3}\s*$", line))

    score += min(dotted_lines, 15)
    score += min(numbered_lines, 15)

    return score


def find_contents_pages(pages: List[Dict], max_scan_pages: int = 20) -> List[int]:
    candidate_scores = []

    for page in pages[:max_scan_pages]:
        page_num = page["page_number"]
        text = page["text"]
        score = score_contents_page(text)

        if score >= 8:
            candidate_scores.append((page_num, score))

    if not candidate_scores:
        return []

    candidate_scores = sorted(candidate_scores, key=lambda x: x[0])

    selected_pages = [candidate_scores[0][0]]
    for page_num, score in candidate_scores[1:]:
        if page_num - selected_pages[-1] <= 2:
            selected_pages.append(page_num)

    return selected_pages


def clean_contents_entry_title(title: str) -> str:
    title = normalize_spaces(title)
    title = re.sub(r"\.{2,}", " ", title)
    title = re.sub(r"\s{2,}", " ", title).strip(" .-–—")
    return title.strip()


def is_valid_contents_title(title: str) -> bool:
    title_l = title.lower().strip()

    if not title or len(title.split()) < 2:
        return False

    bad_titles = {
        "contents",
        "content",
        "index",
        "annual report",
        "annual report 2024-25",
    }

    if title_l in bad_titles:
        return False

    if re.fullmatch(r"[\d\.\-\(\) ]+", title):
        return False

    return True


def is_major_section_title(title: str) -> bool:
    title_l = title.lower().strip()

    major_patterns = [
        r"historical perspective",
        r"board of directors",
        r"performance highlights",
        r"chairman.?s address",
        r"chairman.?s message",
        r"directors.? report",
        r"notice",
        r"csr",
        r"corporate social responsibility",
        r"annual report on csr activities",
        r"management discussion",
        r"management discussions",
        r"management discussion and analysis",
        r"management discussions and analysis",
        r"report on corporate governance",
        r"certificate on corporate governance",
        r"corporate governance",
        r"secretarial audit report",
        r"ceo\s*&\s*cfo certificate",
        r"ceo\s+and\s+cfo certificate",
        r"independent auditor.?s report",
        r"comments of c\s*&\s*ag",
        r"comments of c and ag",
        r"significant accounting policies",
        r"notes forming part of standalone financial statement",
        r"notes forming part of standalone financial statements",
        r"notes forming part of consolidated financial statement",
        r"notes forming part of consolidated financial statements",
        r"standalone financial statement",
        r"standalone financial statements",
        r"consolidated financial statement",
        r"consolidated financial statements",
        r"business responsibility",
        r"sustainability",
        r"brsr",
    ]

    return any(re.search(pattern, title_l) for pattern in major_patterns)


def parse_contents_entries_from_page(text: str) -> List[Dict]:
    entries = []
    lines = [line.strip() for line in text.splitlines() if line.strip()]

    extracted = []

    for line in lines:
        # Case 1: title ...... page_number
        matches = re.findall(r"([A-Za-z][A-Za-z0-9’'&,/\-\(\) ]+?)\s*\.{2,}\s*(\d{1,3})", line)
        for title, page_no in matches:
            extracted.append((clean_contents_entry_title(title), int(page_no)))

        # Case 2: title + multiple spaces + page_number
        matches = re.findall(r"([A-Za-z][A-Za-z0-9’'&,/\-\(\) ]+?)\s{2,}(\d{1,3})(?=\s|$)", line)
        for title, page_no in matches:
            extracted.append((clean_contents_entry_title(title), int(page_no)))

        # Case 3: merged text like "... Company 5 Notice to Shareholders 6 ..."
        matches = re.findall(r"([A-Z][A-Za-z0-9’'&,/\-\(\) ]+?)\s+(\d{1,3})(?=\s+[A-Z]|\s*$)", line)
        for title, page_no in matches:
            extracted.append((clean_contents_entry_title(title), int(page_no)))

        # Case 4: title with explicit page range
        m = re.match(r"^(.*?)\s*\(Pages?\s+(\d{1,3})\s+to\s+(\d{1,3})\)\s*$", line, flags=re.I)
        if m:
            extracted.append((clean_contents_entry_title(m.group(1)), int(m.group(2))))

    seen = set()
    for title, start_page in extracted:
        if not is_valid_contents_title(title):
            continue

        key = (title.lower(), start_page)
        if key in seen:
            continue
        seen.add(key)

        entries.append({
            "title": title,
            "start_page": start_page
        })

    return entries


def parse_contents_entries(pages: List[Dict], contents_page_numbers: List[int]) -> List[Dict]:
    all_entries = []
    page_lookup = {p["page_number"]: p for p in pages}

    for page_num in contents_page_numbers:
        page = page_lookup.get(page_num)
        if not page:
            continue

        page_entries = parse_contents_entries_from_page(page["text"])
        all_entries.extend(page_entries)

    seen = set()
    deduped = []
    for entry in all_entries:
        key = (entry["title"].lower(), entry["start_page"])
        if key not in seen:
            seen.add(key)
            deduped.append(entry)

    deduped = sorted(deduped, key=lambda x: x["start_page"])
    return deduped


def classify_section_type(title: str) -> str:
    title_l = title.lower().strip()

    patterns = [
        (r"historical perspective", "general"),
        (r"board of directors", "leadership"),
        (r"performance highlights", "highlights"),
        (r"chairman.?s address", "leadership_message"),
        (r"chairman.?s message", "leadership_message"),
        (r"directors.? report", "board_report"),
        (r"notice", "notice"),
        (r"csr", "esg"),
        (r"corporate social responsibility", "esg"),
        (r"annual report on csr activities", "esg"),
        (r"management discussion", "management_discussion"),
        (r"management discussions", "management_discussion"),
        (r"report on corporate governance", "governance"),
        (r"certificate on corporate governance", "governance"),
        (r"corporate governance", "governance"),
        (r"secretarial audit report", "audit_report"),
        (r"ceo\s*&\s*cfo certificate", "compliance_certificate"),
        (r"ceo\s+and\s+cfo certificate", "compliance_certificate"),
        (r"independent auditor.?s report", "audit_report"),
        (r"comments of c\s*&\s*ag", "audit_report"),
        (r"comments of c and ag", "audit_report"),
        (r"significant accounting policies", "financial_statements"),
        (r"notes forming part of standalone financial statement", "financial_statements"),
        (r"notes forming part of standalone financial statements", "financial_statements"),
        (r"notes forming part of consolidated financial statement", "financial_statements"),
        (r"notes forming part of consolidated financial statements", "financial_statements"),
        (r"standalone financial statement", "financial_statements"),
        (r"standalone financial statements", "financial_statements"),
        (r"consolidated financial statement", "financial_statements"),
        (r"consolidated financial statements", "financial_statements"),
        (r"business responsibility", "esg"),
        (r"sustainability", "esg"),
        (r"brsr", "esg"),
    ]

    for pattern, section_type in patterns:
        if re.search(pattern, title_l):
            return section_type

    return "general"


def build_sections_from_contents(cleaned_pages: List[Dict], contents_entries: List[Dict]) -> List[Dict]:
    if not contents_entries:
        return []

    total_pages = cleaned_pages[-1]["page_number"] if cleaned_pages else 0
    page_lookup = {p["page_number"]: p["text"] for p in cleaned_pages}

    sections = []

    first_start = contents_entries[0]["start_page"]
    if first_start > 1:
        opening_text_parts = []
        for pnum in range(1, first_start):
            if pnum in page_lookup and page_lookup[pnum].strip():
                opening_text_parts.append(page_lookup[pnum].strip())

        opening_text = "\n\n".join(opening_text_parts).strip()
        if opening_text:
            sections.append({
                "section_id": str(uuid.uuid4()),
                "title": "Document Opening",
                "section_type": "general",
                "page_start": 1,
                "page_end": first_start - 1,
                "text": opening_text
            })

    for idx, entry in enumerate(contents_entries):
        title = entry["title"]
        start_page = entry["start_page"]

        if idx < len(contents_entries) - 1:
            end_page = contents_entries[idx + 1]["start_page"] - 1
        else:
            end_page = total_pages

        if start_page > end_page:
            continue

        text_parts = []
        for pnum in range(start_page, end_page + 1):
            if pnum in page_lookup and page_lookup[pnum].strip():
                text_parts.append(page_lookup[pnum].strip())

        section_text = "\n\n".join(text_parts).strip()
        if not section_text:
            continue

        sections.append({
            "section_id": str(uuid.uuid4()),
            "title": title,
            "section_type": classify_section_type(title),
            "page_start": start_page,
            "page_end": end_page,
            "text": section_text
        })

    return sections


# -------- Detect sections using Contents / Index pages --------
contents_page_numbers = find_contents_pages(contents_pages, max_scan_pages=20)
contents_entries = parse_contents_entries(contents_pages, contents_page_numbers)
contents_entries = [e for e in contents_entries if is_major_section_title(e["title"])]
sections = build_sections_from_contents(cleaned_pages, contents_entries)

print("Detected contents/index pages:", contents_page_numbers)
print("Parsed contents entries:", len(contents_entries))

if contents_page_numbers:
    print("\nRaw contents page preview:\n")
    page_lookup_debug = {p["page_number"]: p for p in contents_pages}
    for pnum in contents_page_numbers:
        print("=" * 100)
        print(f"CONTENTS PAGE {pnum}")
        print("-" * 100)
        print(page_lookup_debug[pnum]["text"][:3000])
        print()

if contents_entries:
    print("Sample parsed entries:")
    for entry in contents_entries[:15]:
        print(f"- {entry['title']} -> {entry['start_page']}")
else:
    print("No contents entries parsed. Falling back to heading-based section detection.")


# -------- Fallback: heading-based section detection --------
if not sections:
    def uppercase_ratio(text: str) -> float:
        letters = [ch for ch in text if ch.isalpha()]
        if not letters:
            return 0.0
        upper = [ch for ch in letters if ch.isupper()]
        return len(upper) / len(letters)

    def looks_like_heading(line: str) -> bool:
        line = line.strip()

        if not line:
            return False

        word_count = len(line.split())

        if word_count < 2 or word_count > 10:
            return False

        if line.endswith(".") or line.endswith(";") or line.endswith(":"):
            return False

        if re.search(r"\d{2,}", line):
            return False

        if line.count(",") > 1:
            return False

        if re.match(r"^(\-|\•|\*|\d+\.)\s+", line):
            return False

        if line.isupper() and 2 <= word_count <= 8:
            return True

        if re.fullmatch(r"[A-Z][A-Za-z&/\-(), ]+", line):
            if uppercase_ratio(line) >= 0.30:
                return True
            if line.istitle():
                return True

        return False

    fallback_sections = []

    current_title = "Document Opening"
    current_type = "general"
    current_page_start = cleaned_pages[0]["page_number"] if cleaned_pages else 1
    current_text_parts = []

    for page in cleaned_pages:
        lines = page["text"].splitlines()
        page_buffer = []

        for line in lines:
            if looks_like_heading(line):
                title_candidate = line.strip()

                if page_buffer:
                    current_text_parts.append("\n".join(page_buffer).strip())
                    page_buffer = []

                existing_text = "\n".join(current_text_parts).strip()
                if existing_text:
                    text = "\n".join(part for part in current_text_parts if part.strip()).strip()
                    if text:
                        fallback_sections.append({
                            "section_id": str(uuid.uuid4()),
                            "title": current_title,
                            "section_type": classify_section_type(current_title),
                            "page_start": current_page_start,
                            "page_end": page["page_number"],
                            "text": text
                        })

                    current_title = title_candidate
                    current_type = classify_section_type(title_candidate)
                    current_page_start = page["page_number"]
                    current_text_parts = []
                else:
                    current_title = title_candidate
                    current_type = classify_section_type(title_candidate)
            else:
                page_buffer.append(line)

        if page_buffer:
            current_text_parts.append("\n".join(page_buffer).strip())

    if cleaned_pages:
        text = "\n".join(part for part in current_text_parts if part.strip()).strip()
        if text:
            fallback_sections.append({
                "section_id": str(uuid.uuid4()),
                "title": current_title,
                "section_type": current_type,
                "page_start": current_page_start,
                "page_end": cleaned_pages[-1]["page_number"],
                "text": text
            })

    sections = fallback_sections


# -------- Validate sections --------
validated_sections = []
for s in sections:
    if not s["title"].strip():
        continue
    if not s["text"].strip():
        continue
    if s["page_start"] > s["page_end"]:
        continue
    validated_sections.append(s)

sections = validated_sections

print("\nFinal sections after validation:", len(sections))
for s in sections[:15]:
    print(f"{s['page_start']:>3}-{s['page_end']:<3} | {s['section_type']:<22} | {s['title']}")


# -------- Chunking --------
def split_into_paragraphs(text: str) -> List[str]:
    return [p.strip() for p in re.split(r"\n\s*\n", text) if p.strip()]

def wc(text: str) -> int:
    return len(text.split())

def chunk_section_text(section_text: str) -> List[Dict]:
    paragraphs = split_into_paragraphs(section_text)

    chunks = []
    current_words = []
    current_parts = []

    def flush():
        nonlocal current_words, current_parts, chunks
        text = "\n\n".join(current_parts).strip()
        if text:
            chunks.append({
                "text": text,
                "word_count": wc(text)
            })
        current_words = []
        current_parts = []

    for para in paragraphs:
        para_wc = wc(para)

        if para_wc > MAX_CHUNK_WORDS:
            if current_parts:
                flush()

            sentences = re.split(r"(?<=[.!?])\s+", para)
            temp_parts = []
            temp_words = []

            for sent in sentences:
                sent_wc = wc(sent)
                if len(temp_words) + sent_wc <= MAX_CHUNK_WORDS:
                    temp_parts.append(sent)
                    temp_words.extend(sent.split())
                else:
                    if temp_parts:
                        text = " ".join(temp_parts).strip()
                        chunks.append({
                            "text": text,
                            "word_count": wc(text)
                        })
                    temp_parts = [sent]
                    temp_words = sent.split()

            if temp_parts:
                text = " ".join(temp_parts).strip()
                chunks.append({
                    "text": text,
                    "word_count": wc(text)
                })
            continue

        if len(current_words) + para_wc <= TARGET_CHUNK_WORDS:
            current_parts.append(para)
            current_words.extend(para.split())
        else:
            if len(current_words) < MIN_CHUNK_WORDS and len(current_words) + para_wc <= MAX_CHUNK_WORDS:
                current_parts.append(para)
                current_words.extend(para.split())
            else:
                flush()
                current_parts.append(para)
                current_words.extend(para.split())

    if current_parts:
        flush()

    if CHUNK_OVERLAP_WORDS > 0 and len(chunks) > 1:
        new_chunks = []
        for i, ch in enumerate(chunks):
            if i == 0:
                new_chunks.append(ch)
                continue

            prev_words = chunks[i - 1]["text"].split()
            overlap_words = prev_words[-CHUNK_OVERLAP_WORDS:] if len(prev_words) >= CHUNK_OVERLAP_WORDS else prev_words
            new_text = " ".join(overlap_words) + "\n\n" + ch["text"]
            new_chunks.append({
                "text": new_text.strip(),
                "word_count": wc(new_text)
            })
        chunks = new_chunks

    return chunks


chunks = []

for section in sections:
    section_chunks = chunk_section_text(section["text"])

    for ch in section_chunks:
        if not ch["text"].strip():
            continue

        chunks.append({
            "chunk_id": str(uuid.uuid4()),
            "document_id": phase1_output["document_id"],
            "document_name": phase1_output["document_name"],
            "company_name": phase1_output["company_name"],
            "financial_year": phase1_output["financial_year"],
            "report_title": phase1_output["report_title"],
            "section_id": section["section_id"],
            "section_name": section["title"],
            "section_type": section["section_type"],
            "page_start": section["page_start"],
            "page_end": section["page_end"],
            "word_count": ch["word_count"],
            "chunk_text": ch["text"]
        })

chunks = [c for c in chunks if c["chunk_text"].strip() and c["word_count"] > 0]

print("=" * 100)
print("PHASE 2 COMPLETE")
print("=" * 100)
print("Document Name  :", phase1_output["document_name"])
print("Document ID    :", phase1_output["document_id"])
print("Company Name   :", phase1_output["company_name"])
print("Financial Year :", phase1_output["financial_year"])
print("Total Sections :", len(sections))
print("Total Chunks   :", len(chunks))

print("\nFirst Chunk Metadata")
print("Chunk ID       :", chunks[0]["chunk_id"])
print("Section Name   :", chunks[0]["section_name"])
print("Section Type   :", chunks[0]["section_type"])
print("Page Range     :", f'{chunks[0]["page_start"]} - {chunks[0]["page_end"]}')
print("Word Count     :", chunks[0]["word_count"])

print("\nFirst Chunk Preview\n")
print(chunks[0]["chunk_text"][:2000])

phase2_sections_path = NOTEBOOK_RECORDS_DIR / "phase2_sections.json"
phase2_chunks_path = NOTEBOOK_RECORDS_DIR / "phase2_chunks.json"
phase2_chunks_csv_path = NOTEBOOK_RECORDS_DIR / "phase2_chunks_review.csv"

with open(phase2_sections_path, "w", encoding="utf-8") as f:
    json.dump(sections, f, ensure_ascii=False, indent=2)

with open(phase2_chunks_path, "w", encoding="utf-8") as f:
    json.dump(chunks, f, ensure_ascii=False, indent=2)

pd.DataFrame(chunks).to_csv(phase2_chunks_csv_path, index=False, encoding="utf-8")

print("\nPhase 2 outputs saved to:")
print("-", phase2_sections_path)
print("-", phase2_chunks_path)
print("-", phase2_chunks_csv_path)

print("\nSection Type Distribution")
print(pd.DataFrame(sections)["section_type"].value_counts(dropna=False))

print("\nChunk Word Count Summary")
print(pd.DataFrame(chunks)["word_count"].describe())

Detected contents/index pages: [9]
Parsed contents entries: 16

Raw contents page preview:

CONTENTS PAGE 9
----------------------------------------------------------------------------------------------------
VII
Annual Report 2024 - 25
CONTENTS
HMT LIMITED
Board of Directors.....................................................................................................................1
Performance Highlights.............................................................................................................2
Directors’ Report.......................................................................................................................3
Annual Report on CSR Activities.............................................................................................14
Management Discussions and Analysis..................................................................................19
Report on Corporate Governance.........................................................

In [ ]:
#Add a quick Phase 2 quality summary

print("\nSection Type Distribution")
print(pd.DataFrame(sections)["section_type"].value_counts(dropna=False))

print("\nChunk Word Count Summary")
print(pd.DataFrame(chunks)["word_count"].describe())

In [8]:
# ==========================================
# PHASE 3 — EMBEDDINGS + RETRIEVAL
# ==========================================

chunks_df = pd.DataFrame(chunks).copy()

print("Chunks shape:", chunks_df.shape)
display(chunks_df.head(3))

Chunks shape: (486, 13)


,chunk_id,document_id,document_name,company_name,financial_year,report_title,section_id,section_name,section_type,page_start,page_end,word_count,chunk_text
0,4ef77baa-815a-425b-8cbd-b9c513bb1a0e,cac28f9d866d468b9b958ee01d9366e6,Annual-Report-2024-25.pdf,HMT Limited,2024-25,Annual Report 2024 - 25,485cb01e-71ea-4955-afdc-bf9411ce2830,Directors’ Report,board_report,3,13,258,I\nAnnual Report 2024 - 25 Chairman’s Address ...
1,e386b3b5-fc8b-46ba-bf97-4417db70e4ff,cac28f9d866d468b9b958ee01d9366e6,Annual-Report-2024-25.pdf,HMT Limited,2024-25,Annual Report 2024 - 25,485cb01e-71ea-4955-afdc-bf9411ce2830,Directors’ Report,board_report,3,13,286,"workforce. Inflation is moderating, providing ..."
2,612a15c8-c203-47e8-a1f2-6dbd3f86d1f3,cac28f9d866d468b9b958ee01d9366e6,Annual-Report-2024-25.pdf,HMT Limited,2024-25,Annual Report 2024 - 25,485cb01e-71ea-4955-afdc-bf9411ce2830,Directors’ Report,board_report,3,13,281,it well to remain among the fastest-growing la...


In [9]:
def build_embedding_text(row) -> str:
    parts = [
        f"Company: {row['company_name']}",
        f"Report Title: {row['report_title']}",
        f"Financial Year: {row['financial_year']}",
        f"Section Name: {row['section_name']}",
        f"Section Type: {row['section_type']}",
        f"Page Range: {row['page_start']} to {row['page_end']}",
        "Content:",
        row["chunk_text"]
    ]
    return "\n".join(parts)

chunks_df["embedding_text"] = chunks_df.apply(build_embedding_text, axis=1)

print("Embedding text sample:\n")
print(chunks_df.loc[0, "embedding_text"][:2500])

Embedding text sample:

Company: HMT Limited
Report Title: Annual Report 2024 - 25
Financial Year: 2024-25
Section Name: Directors’ Report
Section Type: board_report
Page Range: 3 to 13
Content:
I
Annual Report 2024 - 25 Chairman’s Address 72nd Annual General Meeting of HMT Limited My Dear Shareholders, I am honored to present to you the Annual Report of HMT Limited for the year 2024-25. We shall reflect on the past year’s achievements and challenges. It is also imperative to acknowledge the global outlook, the state of the Indian Economy, and the impact of manufacturing initiatives on the future of our nation. Global Outlook The global economy is entering a phase of modest but uneven expansion. According to the IMF’s World Economic Outlook (April 2025), world GDP growth is expected to edge up slightly from 3.2 percent in 2024 to 3.3 percent in 2025, with headline inflation continuing its gradual decline despite lingering upside risks from trade tensions and tighter monetary policy in 

In [10]:
#Load Embedding Model
EMBEDDING_MODEL_NAME = "all-MiniLM-L6-v2"

print("Loading embedding model:", EMBEDDING_MODEL_NAME)
embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)
print("Model loaded successfully.")

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 13965.20it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded successfully.


In [11]:
#Generate chunk embeddings

chunk_texts_for_embedding = chunks_df["embedding_text"].tolist()

chunk_embeddings = embedding_model.encode(
    chunk_texts_for_embedding,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print("Chunk embeddings shape:", chunk_embeddings.shape)

Batches: 100%|██████████| 16/16 [00:05<00:00,  3.03it/s]

Chunk embeddings shape: (486, 384)


In [12]:
#Save embedding artifacts

phase3_embeddings_path = NOTEBOOK_RECORDS_DIR / "phase3_chunk_embeddings.npy"
phase3_chunks_enriched_path = NOTEBOOK_RECORDS_DIR / "phase3_chunks_enriched.csv"

np.save(phase3_embeddings_path, chunk_embeddings)
chunks_df.to_csv(phase3_chunks_enriched_path, index=False, encoding="utf-8")

print("Phase 3 embedding artifacts saved:")
print("-", phase3_embeddings_path)
print("-", phase3_chunks_enriched_path)

Phase 3 embedding artifacts saved:
- /home/yash/Dev_Projects/notebook_records/phase3_chunk_embeddings.npy
- /home/yash/Dev_Projects/notebook_records/phase3_chunks_enriched.csv


In [13]:
#Query embedding helper

def embed_query(query: str) -> np.ndarray:
    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    )
    return query_embedding[0]

In [14]:
#Simple retrieval

def retrieve_top_k_chunks(query: str, top_k: int = TOP_K) -> pd.DataFrame:
    query_vec = embed_query(query)
    scores = cosine_similarity([query_vec], chunk_embeddings)[0]

    result_df = chunks_df.copy()
    result_df["retrieval_score"] = scores

    result_df = result_df.sort_values(
        by="retrieval_score",
        ascending=False
    ).head(top_k).reset_index(drop=True)

    return result_df

In [15]:
#First retrieval test

test_query = "What are the main CSR activities carried out by the company?"

top_results = retrieve_top_k_chunks(test_query, top_k=5)

display(
    top_results[
        [
            "chunk_id",
            "section_name",
            "section_type",
            "page_start",
            "page_end",
            "word_count",
            "retrieval_score"
        ]
    ]
)

,chunk_id,section_name,section_type,page_start,page_end,word_count,retrieval_score
0,c20b9834-a0e4-40ff-a909-0b125a1231e6,Report on Corporate Governance,governance,23,39,174,0.633742
1,40f046f7-f731-4a6d-b614-3519b6820521,Report on Corporate Governance,governance,23,39,285,0.593922
2,13bc30d1-fdf8-4899-839e-ff2c8a979089,Report on Corporate Governance,governance,23,39,264,0.577060
3,2b97bac3-383e-4c17-a3a9-6795e821d42d,Report on Corporate Governance,governance,23,39,216,0.572055
4,f9d83257-a640-49a4-9891-04d4b6ab67f9,Report on Corporate Governance,governance,23,39,127,0.547616


In [16]:
#Retrieval preview helper

def show_retrieval_results(query: str, top_k: int = 5, preview_chars: int = 1200):
    results = retrieve_top_k_chunks(query, top_k=top_k)

    print("=" * 120)
    print("QUERY:", query)
    print("=" * 120)

    for idx, row in results.iterrows():
        print(f"\nRank #{idx + 1}")
        print("-" * 120)
        print("Score       :", round(row["retrieval_score"], 4))
        print("Section     :", row["section_name"])
        print("Type        :", row["section_type"])
        print("Pages       :", f'{row["page_start"]} - {row["page_end"]}')
        print("Word Count  :", row["word_count"])
        print("Chunk ID    :", row["chunk_id"])
        print("\nPreview:\n")
        print(row["chunk_text"][:preview_chars])
        print("\n")

In [17]:
#Few tests

show_retrieval_results("What are the main CSR activities carried out by the company?", top_k=5)
show_retrieval_results("Summarize the management discussion and analysis section.", top_k=5)
show_retrieval_results("What does the report say about corporate governance?", top_k=5)
show_retrieval_results("What observations are mentioned in the audit report?", top_k=5)
show_retrieval_results("What do the financial statements say about the company?", top_k=5)

QUERY: What are the main CSR activities carried out by the company?

Rank #1
------------------------------------------------------------------------------------------------------------------------
Score       : 0.6337
Section     : Report on Corporate Governance
Type        : governance
Pages       : 23 - 39
Word Count  : 174
Chunk ID    : c20b9834-a0e4-40ff-a909-0b125a1231e6

Preview:

undertaken by the Company as specified in Schedule VII of the Companies Act and the expenditure thereon, excluding activities undertaken in pursuance of normal course of business of the Company.

c) The CSR projects or programs or activities that benefit only the employees of the company and their families shall not be considered as CSR activities in accordance with section 135 of the Act. d) The CSR and Sustainability budget expenditure shall be fixed in accordance with the provisions of the Act, Rules and the Guidelines. e) The Company will endeavor at all times to build and develop the skills of its

In [22]:
def infer_section_filter_from_query(query: str):
    q = query.lower()

    section_map = {
        "esg": ["csr", "sustainability", "esg", "social responsibility", "brsr"],
        "management_discussion": ["management discussion", "management analysis", "md&a", "outlook", "performance review"],
        "governance": ["corporate governance", "governance"],
        "audit_report": ["audit", "auditor", "secretarial audit", "independent auditor"],
        "financial_statements": ["financial statements", "balance sheet", "profit and loss", "cash flow", "assets", "liabilities", "notes to accounts"],
        "board_report": ["directors report", "director's report", "board report"],
        "compliance_certificate": ["ceo cfo certificate", "compliance certificate", "certificate"],
        "leadership_message": ["chairman message", "chairman's message", "chairman address"],
        "highlights": ["performance highlights", "highlights"]
    }

    matched_types = []
    for section_type, keywords in section_map.items():
        if any(keyword in q for keyword in keywords):
            matched_types.append(section_type)

    return matched_types if matched_types else None


def retrieve_top_k_chunks_filtered(query: str, top_k: int = TOP_K) -> pd.DataFrame:
    query_vec = embed_query(query)
    scores = cosine_similarity([query_vec], chunk_embeddings)[0]

    result_df = chunks_df.copy()
    result_df["retrieval_score"] = scores

    section_filters = infer_section_filter_from_query(query)

    if section_filters:
        filtered_df = result_df[result_df["section_type"].isin(section_filters)].copy()
        if not filtered_df.empty:
            result_df = filtered_df

    result_df = result_df.sort_values("retrieval_score", ascending=False).head(top_k).reset_index(drop=True)
    return result_df


def show_retrieval_results_filtered(query: str, top_k: int = 5, preview_chars: int = 1200):
    section_filters = infer_section_filter_from_query(query)
    results = retrieve_top_k_chunks_filtered(query, top_k=top_k)

    print("=" * 120)
    print("QUERY:", query)
    print("SECTION FILTERS:", section_filters)
    print("=" * 120)

    for idx, row in results.iterrows():
        print(f"\nRank #{idx + 1}")
        print("-" * 120)
        print("Score       :", round(row["retrieval_score"], 4))
        print("Section     :", row["section_name"])
        print("Type        :", row["section_type"])
        print("Pages       :", f"{row['page_start']} - {row['page_end']}")
        print("Word Count  :", row["word_count"])
        print("\nPreview:\n")
        print(row["chunk_text"][:preview_chars])
        print("\n")

In [23]:
show_retrieval_results_filtered(
    "What are the main CSR activities carried out by the company?",
    top_k=5
)

QUERY: What are the main CSR activities carried out by the company?
SECTION FILTERS: ['esg']

Rank #1
------------------------------------------------------------------------------------------------------------------------
Score       : 0.5222
Section     : Annual Report on CSR Activities
Type        : esg
Pages       : 14 - 18
Word Count  : 252

Preview:

complied with provisions relating to the constitution of Internal Complaints Committee under the Sexual Harassment of Women at Workplace (Prevention, Prohibition and Redressal) Act, 2013 and following information are submitted.

i. Number of complaints of sexual harassment received in the year - NIL ii. Number of complaints disposed off during the year - NIL iii. Number of cases pending for more than ninety days – NIL
FRAUD REPORTING
There was no incident of fraud reported during the year under review. CORPORATE SOCIAL RESPONSIBILITY (CSR)
The Board level CSR Committee was constituted on 12th August 2019. The composition of the CSR C

In [19]:
# #Display Helper

# def show_retrieval_results_filtered(query: str, top_k: int = 5, preview_chars: int = 1200):
#     section_filters = infer_section_filter_from_query(query)
#     results = retrieve_top_k_chunks_filtered(query, top_k=top_k)

#     print("=" * 120)
#     print("QUERY:", query)
#     print("SECTION FILTERS:", section_filters)
#     print("=" * 120)

#     for idx, row in results.iterrows():
#         print(f"\nRank #{idx + 1}")
#         print("-" * 120)
#         print("Score       :", round(row["retrieval_score"], 4))
#         print("Section     :", row["section_name"])
#         print("Type        :", row["section_type"])
#         print("Pages       :", f'{row["page_start"]} - {row["page_end"]}')
#         print("Word Count  :", row["word_count"])
#         print("\nPreview:\n")
#         print(row["chunk_text"][:preview_chars])
#         print("\n")

In [24]:
#Top-k retrieval testing

sample_queries = [
    "What are the main CSR activities carried out by the company?",
    "Summarize the management discussion and analysis section.",
    "What does the report say about corporate governance?",
    "What observations are mentioned in the secretarial audit report?",
    "What does the CEO and CFO certificate mention?",
    "What are the major points in the directors report?",
    "What do the financial statements indicate about the company?",
    "What are the key performance highlights?",
]

In [25]:
#Test each

for query in sample_queries:
    show_retrieval_results_filtered(query, top_k=5, preview_chars=700)

QUERY: What are the main CSR activities carried out by the company?
SECTION FILTERS: ['esg']

Rank #1
------------------------------------------------------------------------------------------------------------------------
Score       : 0.5222
Section     : Annual Report on CSR Activities
Type        : esg
Pages       : 14 - 18
Word Count  : 252

Preview:

complied with provisions relating to the constitution of Internal Complaints Committee under the Sexual Harassment of Women at Workplace (Prevention, Prohibition and Redressal) Act, 2013 and following information are submitted.

i. Number of complaints of sexual harassment received in the year - NIL ii. Number of complaints disposed off during the year - NIL iii. Number of cases pending for more than ninety days – NIL
FRAUD REPORTING
There was no incident of fraud reported during the year under review. CORPORATE SOCIAL RESPONSIBILITY (CSR)
The Board level CSR Committee was constituted on 12th August 2019. The composition of the CSR C

In [26]:
#Evaluate Hit@k and MRR

eval_questions = [
    {
        "question": "What are the main CSR activities carried out by the company?",
        "expected_section_type": "esg"
    },
    {
        "question": "Summarize the management discussion and analysis section.",
        "expected_section_type": "management_discussion"
    },
    {
        "question": "What does the report say about corporate governance?",
        "expected_section_type": "governance"
    },
    {
        "question": "What observations are mentioned in the secretarial audit report?",
        "expected_section_type": "audit_report"
    },
    {
        "question": "What does the CEO and CFO certificate mention?",
        "expected_section_type": "compliance_certificate"
    },
    {
        "question": "What are the major points in the directors report?",
        "expected_section_type": "board_report"
    },
    {
        "question": "What do the financial statements indicate about the company?",
        "expected_section_type": "financial_statements"
    },
    {
        "question": "What are the key performance highlights?",
        "expected_section_type": "highlights"
    }
]

eval_df = pd.DataFrame(eval_questions)
display(eval_df)

,question,expected_section_type
0,What are the main CSR activities carried out b...,esg
1,Summarize the management discussion and analys...,management_discussion
2,What does the report say about corporate gover...,governance
3,What observations are mentioned in the secreta...,audit_report
4,What does the CEO and CFO certificate mention?,compliance_certificate
5,What are the major points in the directors rep...,board_report
6,What do the financial statements indicate abou...,financial_statements
7,What are the key performance highlights?,highlights


In [28]:
# ==========================================
# PHASE 3 — RETRIEVAL EVALUATION
# ==========================================

def evaluate_single_query(query: str, expected_section_type: str, top_k: int = 5):
    results = retrieve_top_k_chunks_filtered(query, top_k=top_k)

    matches = (results["section_type"] == expected_section_type).tolist()

    hit_at_k = int(any(matches))

    reciprocal_rank = 0.0
    for idx, is_match in enumerate(matches, start=1):
        if is_match:
            reciprocal_rank = 1.0 / idx
            break

    return {
        "question": query,
        "expected_section_type": expected_section_type,
        "top_k": top_k,
        "hit_at_k": hit_at_k,
        "mrr": reciprocal_rank,
        "top_section_types": results["section_type"].tolist(),
        "top_section_names": results["section_name"].tolist(),
        "top_scores": [round(x, 4) for x in results["retrieval_score"].tolist()]
    }


def evaluate_across_k(eval_df, k_values=[1, 3, 5, 10]):
    rows = []

    for k in k_values:
        temp_results = []

        for _, row in eval_df.iterrows():
            result = evaluate_single_query(
                query=row["question"],
                expected_section_type=row["expected_section_type"],
                top_k=k
            )
            temp_results.append(result)

        temp_df = pd.DataFrame(temp_results)

        rows.append({
            "k": k,
            "hit_rate": temp_df["hit_at_k"].mean(),
            "mrr": temp_df["mrr"].mean()
        })

    return pd.DataFrame(rows)

In [29]:
#Run

eval_results = []

for _, row in eval_df.iterrows():
    result = evaluate_single_query(
        query=row["question"],
        expected_section_type=row["expected_section_type"],
        top_k=5
    )
    eval_results.append(result)

eval_results_df = pd.DataFrame(eval_results)
display(eval_results_df)

,question,expected_section_type,top_k,hit_at_k,mrr,top_section_types,top_section_names,top_scores
0,What are the main CSR activities carried out b...,esg,5,1,1.0,"[esg, esg, esg, esg, esg]","[Annual Report on CSR Activities, Annual Repor...","[0.5222, 0.4335, 0.4288, 0.4077, 0.4047]"
1,Summarize the management discussion and analys...,management_discussion,5,1,1.0,"[management_discussion, management_discussion,...","[Management Discussions and Analysis, Manageme...","[0.4246, 0.4109, 0.3757, 0.3755, 0.3659]"
2,What does the report say about corporate gover...,governance,5,1,1.0,"[governance, governance, governance, governanc...","[Report on Corporate Governance, Report on Cor...","[0.642, 0.588, 0.575, 0.5624, 0.5518]"
3,What observations are mentioned in the secreta...,audit_report,5,1,1.0,"[audit_report, audit_report, audit_report, aud...","[Independent Auditor’s Report, Independent Aud...","[0.59, 0.5891, 0.5818, 0.5593, 0.5484]"
4,What does the CEO and CFO certificate mention?,compliance_certificate,5,1,1.0,"[compliance_certificate, compliance_certificat...","[CEO & CFO Certificate, CEO & CFO Certificate,...","[0.5177, 0.5158, 0.4465, 0.4304]"
5,What are the major points in the directors rep...,board_report,5,1,1.0,"[board_report, board_report, board_report, boa...","[Directors’ Report, Directors’ Report, Directo...","[0.5125, 0.5116, 0.4419, 0.3519, 0.317]"
6,What do the financial statements indicate abou...,financial_statements,5,1,1.0,"[financial_statements, financial_statements, f...","[Consolidated Financial Statement, Consolidate...","[0.6263, 0.6223, 0.6177, 0.6169, 0.6152]"
7,What are the key performance highlights?,highlights,5,0,0.0,"[board_report, financial_statements, esg, boar...","[Directors’ Report, Consolidated Financial Sta...","[0.2552, 0.2298, 0.2226, 0.2126, 0.2098]"


In [30]:
print("Hit@5 :", round(eval_results_df["hit_at_k"].mean(), 4))
print("MRR@5 :", round(eval_results_df["mrr"].mean(), 4))

Hit@5 : 0.875
MRR@5 : 0.875


In [31]:
#And then compare across k:

k_comparison_df = evaluate_across_k(eval_df, k_values=[1, 3, 5, 10])
display(k_comparison_df)

,k,hit_rate,mrr
0,1,0.875,0.875
1,3,0.875,0.875
2,5,0.875,0.875
3,10,0.875,0.875


In [ ]:
#Compare different k values

# def evaluate_across_k(eval_df, k_values=[1, 3, 5, 10]):
#     rows = []

#     for k in k_values:
#         temp_results = []
#         for _, row in eval_df.iterrows():
#             result = evaluate_single_query(
#                 query=row["question"],
#                 expected_section_type=row["expected_section_type"],
#                 top_k=k
#             )
#             temp_results.append(result)

#         temp_df = pd.DataFrame(temp_results)

#         rows.append({
#             "k": k,
#             "hit_rate": temp_df["hit_at_k"].mean(),
#             "mrr": temp_df["mrr"].mean()
#         })

#     return pd.DataFrame(rows)

# k_comparison_df = evaluate_across_k(eval_df, k_values=[1, 3, 5, 10])
# display(k_comparison_df)

In [ ]:
#Save evaluation outputs

phase3_eval_results_path = NOTEBOOK_RECORDS_DIR / "phase3_eval_results.csv"
phase3_k_comparison_path = NOTEBOOK_RECORDS_DIR / "phase3_k_comparison.csv"

eval_results_df.to_csv(phase3_eval_results_path, index=False, encoding="utf-8")
k_comparison_df.to_csv(phase3_k_comparison_path, index=False, encoding="utf-8")

print("Phase 3 evaluation outputs saved:")
print("-", phase3_eval_results_path)
print("-", phase3_k_comparison_path)

In [ ]:
#Optional answer generation preparation

def build_context_for_answer(query: str, top_k: int = 5, max_chars: int = 7000) -> str:
    results = retrieve_top_k_chunks_filtered(query, top_k=top_k)

    blocks = []
    total_chars = 0

    for idx, row in results.iterrows():
        block = f"""
[Chunk {idx + 1}]
Section Name: {row['section_name']}
Section Type: {row['section_type']}
Pages: {row['page_start']} - {row['page_end']}
Content:
{row['chunk_text']}
""".strip()

        if total_chars + len(block) > max_chars:
            break

        blocks.append(block)
        total_chars += len(block)

    return "\n\n".join(blocks)

In [ ]:
#Test

query = "What are the main CSR activities carried out by the company?"
context_text = build_context_for_answer(query, top_k=5)

print(context_text[:5000])

In [ ]:
#Prompt template for answer generation

def build_rag_prompt(query: str, top_k: int = 5) -> str:
    context = build_context_for_answer(query, top_k=top_k)

    prompt = f"""
You are a financial document assistant answering questions from an annual report.

Instructions:
- Answer only from the provided context.
- Do not make up facts.
- If the answer is not clearly available, say so.
- Prefer concise, factual answers.
- Mention section name and page range where relevant.

Question:
{query}

Retrieved Context:
{context}

Answer:
""".strip()

    return prompt

In [ ]:
#Prompt preview
print(build_rag_prompt("What are the main CSR activities carried out by the company?", top_k=5)[:6000])